# Task 2: Trích xuất tín hiệu rPPG và Các thuật toán Baseline

Jupyter Notebook này thực hiện quy trình đọc dữ liệu tiền xử lý của Task 1, trích xuất tín hiệu màu RGB trung bình từ các vùng quan tâm (ROI), áp dụng các thuật toán baseline rPPG (Green Channel, CHROM, POS), và đánh giá kết quả ước lượng nhịp tim (Heart Rate) so với nhãn gốc (Ground Truth) của tập dữ liệu UBFC.

In [ ]:
# Cell 1 - Cài đặt các gói cần thiết (Tương thích với Google Colab)
import sys
import subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print('Đang chạy trên Google Colab. Tiến hành cài đặt các thư viện phụ thuộc...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'scipy', 'opencv-python-headless', 'matplotlib', 'pandas', 'tqdm'])

import json
import re
from pathlib import Path
from typing import Any
import cv2
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as sig
import scipy.signal.windows as windows
from scipy.interpolate import interp1d
from scipy.stats import pearsonr
from tqdm.auto import tqdm

print('Đã nạp toàn bộ các thư viện thành công.')

In [ ]:
# Cell 2 - Cấu hình đường dẫn và các tham số (Hỗ trợ biến môi trường + fallback)
import os
from pathlib import Path

# Xác định thư mục gốc của dự án (PROJECT_ROOT)
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd

# Đọc cấu hình từ biến môi trường hoặc dùng mặc định
NPZ_PATH = os.environ.get('NPZ_PATH', 'outputs/task1_preprocessing/DATASET_1/11-gt/vid_roi_data.npz')
VIDEO_PATH = os.environ.get('VIDEO_PATH', 'datasets/UBFC_DATASET/DATASET_1/11-gt/vid.avi')
GROUND_TRUTH_PATH = os.environ.get('GROUND_TRUTH_PATH', 'datasets/UBFC_DATASET/DATASET_1/11-gt/gtdump.xmp')
OUTPUT_DIR = os.environ.get('OUTPUT_DIR', 'outputs/task2_signals/')

WINDOW_SEC = float(os.environ.get('WINDOW_SEC', 30.0))
STEP_SEC = float(os.environ.get('STEP_SEC', 1.0))

# Phân giải các đường dẫn tương đối so với thư mục gốc dự án
NPZ_PATH = (PROJECT_ROOT / NPZ_PATH).resolve() if not Path(NPZ_PATH).is_absolute() else Path(NPZ_PATH).resolve()
VIDEO_PATH = (PROJECT_ROOT / VIDEO_PATH).resolve() if not Path(VIDEO_PATH).is_absolute() else Path(VIDEO_PATH).resolve()
GROUND_TRUTH_PATH = (PROJECT_ROOT / GROUND_TRUTH_PATH).resolve() if not Path(GROUND_TRUTH_PATH).is_absolute() else Path(GROUND_TRUTH_PATH).resolve()
OUTPUT_DIR = (PROJECT_ROOT / OUTPUT_DIR).resolve() if not Path(OUTPUT_DIR).is_absolute() else Path(OUTPUT_DIR).resolve()

# Tạo thư mục đầu ra nếu chưa tồn tại
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Cấu hình chạy:')
print(f'  - Thư mục gốc dự án: {PROJECT_ROOT}')
print(f'  - File NPZ đầu vào của Task 1: {NPZ_PATH}')
print(f'  - File Video nguồn đầu vào: {VIDEO_PATH}')
print(f'  - File Ground Truth đầu vào: {GROUND_TRUTH_PATH}')
print(f'  - Thư mục lưu kết quả: {OUTPUT_DIR}')
print(f'  - Độ dài cửa sổ FFT: {WINDOW_SEC} giây')
print(f'  - Bước trượt cửa sổ FFT: {STEP_SEC} giây')

In [ ]:
# Cell 3 - Tải các tệp dữ liệu kết quả từ Task 1
if not NPZ_PATH.exists():
    raise FileNotFoundError(f'Không tìm thấy file NPZ tại: {NPZ_PATH}. Hãy chắc chắn đã cấu hình đúng.')

# Tìm file JSON metadata đi kèm
metadata_path = NPZ_PATH.parent / f"{NPZ_PATH.name.replace('_roi_data.npz', '')}_metadata.json"
if not metadata_path.exists():
    metadata_path = NPZ_PATH.parent / 'vid_metadata.json'

print(f'Đang tải dữ liệu NPZ từ {NPZ_PATH}...')
npz_data = dict(np.load(NPZ_PATH, allow_pickle=True))

metadata = {}
if metadata_path.exists():
    print(f'Đang tải metadata từ {metadata_path}...')
    with open(metadata_path, 'r', encoding='utf-8') as f:
        metadata = json.load(f)
    print('Đã tải xong metadata.')
else:
    print('Cảnh báo: Không tìm thấy file metadata.json.')

# Kiểm tra thông tin các mảng dữ liệu đã lưu
print('Thông tin các mảng trong file NPZ:')
for k, v in npz_data.items():
    if isinstance(v, np.ndarray):
        print(f'  - {k}: shape={v.shape}, dtype={v.dtype}')

valid = npz_data['valid']
n_total = len(valid)
n_valid = np.sum(valid)
print(f'\nThống kê khung hình:')
print(f'  - Tổng khung hình: {n_total}')
print(f'  - Khung hình hợp lệ: {n_valid} ({n_valid/n_total*100:.2f}%)')
print(f'  - Khung hình lỗi: {n_total - n_valid}')

In [ ]:
# Cell 4 - Trích xuất tín hiệu màu RGB trung bình từ video nguồn
def extract_roi_rgb(video_path, npz_data, metadata, progress_bar=True):
    """Trích xuất tín hiệu màu RGB trung bình cho từng ROI từ video gốc dựa trên mặt nạ nhị phân."""
    video_path = Path(video_path)
    frame_indices = npz_data['frame_indices']
    roi_masks = npz_data['roi_masks']
    valid = npz_data['valid']

    valid_idxs = np.where(valid)[0]

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f'Không thể mở video nguồn tại: {video_path}')

    fps = metadata['video']['fps']
    mask_h, mask_w = roi_masks.shape[2:]

    n_valid = len(valid_idxs)
    mean_rgb = np.zeros((n_valid, 3, 3), dtype=np.float32)
    timestamps = np.zeros(n_valid, dtype=np.float32)

    success_count = 0
    iterator = range(n_valid)
    if progress_bar:
        iterator = tqdm(iterator, desc='Trích xuất màu RGB cho các ROI', unit='khung hình')

    for i in iterator:
        orig_idx = valid_idxs[i]
        frame_idx = frame_indices[orig_idx]

        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame_bgr = cap.read()

        if not ret:
            continue

        h, w = frame_bgr.shape[:2]
        if w != mask_w or h != mask_h:
            frame_bgr = cv2.resize(frame_bgr, (mask_w, mask_h), interpolation=cv2.INTER_AREA)

        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

        for roi_idx in range(3):
            mask = roi_masks[orig_idx, roi_idx]
            if mask.sum() > 0:
                mean_rgb[success_count, roi_idx] = frame_rgb[mask > 0].mean(axis=0)
            else:
                mean_rgb[success_count, roi_idx] = np.array([np.nan, np.nan, np.nan])

        timestamps[success_count] = frame_idx / fps
        success_count += 1

    cap.release()
    return mean_rgb[:success_count], timestamps[:success_count]

# Chạy trích xuất thực tế hoặc dùng bộ giả lập demo (nếu thiếu file video)
if VIDEO_PATH.exists():
    mean_rgb, timestamps = extract_roi_rgb(VIDEO_PATH, npz_data, metadata)
    print(f'Trích xuất thành công! mean_rgb shape: {mean_rgb.shape}, timestamps shape: {timestamps.shape}')
else:
    print(f'Cảnh báo: Không tìm thấy video nguồn tại {VIDEO_PATH}.')
    print('Đang tạo lập dữ liệu giả cho RGB và mốc thời gian để chạy thử nghiệm...')
    n_valid = np.sum(npz_data['valid'])
    mean_rgb = np.random.normal(120.0, 10.0, size=(n_valid, 3, 3)).astype(np.float32)
    fps = metadata.get('video', {}).get('fps', 30.0)
    t_arr = np.arange(n_valid) / fps
    pulse_wave = np.sin(2 * np.pi * 1.16 * t_arr) * 2.0
    mean_rgb[:, :, 1] += pulse_wave[:, np.newaxis]
    timestamps = t_arr
    print(f'Đã khởi tạo xong. mean_rgb shape: {mean_rgb.shape}')

In [ ]:
# Cell 5 - Hàm tiền xử lý tín hiệu
def preprocess_signal(signal_in, fps, detrend=True, normalize=True, bandpass=(0.7, 3.5)):
    """Tiền xử lý chuỗi tín hiệu thời gian (detrend tuyến tính, lọc Butter thông dải, chuẩn hóa z-score)"""
    if len(signal_in) == 0:
        return signal_in
    
    proc_sig = np.array(signal_in, dtype=np.float32)

    # Khử các giá trị NaN nếu có
    nan_mask = np.isnan(proc_sig)
    if np.any(nan_mask):
        non_nan_indices = np.where(~nan_mask)[0]
        if len(non_nan_indices) > 0:
            indices = np.arange(len(proc_sig))
            proc_sig[nan_mask] = np.interp(indices[nan_mask], non_nan_indices, proc_sig[~nan_mask])
        else:
            return np.zeros_like(proc_sig)

    # Detrend tuyến tính
    if detrend and len(proc_sig) > 1:
        proc_sig = sig.detrend(proc_sig, type='linear')

    # Lọc thông dải (42-210 BPM)
    if bandpass is not None and len(proc_sig) > 9:
        low, high = bandpass
        nyquist = 0.5 * fps
        low_norm = max(0.001, min(0.999, low / nyquist))
        high_norm = max(0.002, min(0.999, high / nyquist))
        b, a = sig.butter(4, [low_norm, high_norm], btype='bandpass')
        proc_sig = sig.filtfilt(b, a, proc_sig)

    # Chuẩn hóa z-score
    if normalize:
        std = np.std(proc_sig)
        if std > 0:
            proc_sig = (proc_sig - np.mean(proc_sig)) / std
        else:
            proc_sig = proc_sig - np.mean(proc_sig)

    return proc_sig

print('Đã biên dịch hàm preprocess_signal.')

In [ ]:
# Cell 6 - Baseline 1: Green channel
def green_channel(mean_rgb, roi_idx=0, fps=30.0, return_rois=False):
    """Green channel baseline method."""
    n_frames, n_rois, _ = mean_rgb.shape
    if return_rois:
        rppg_rois = []
        for i in range(n_rois):
            sig_roi = preprocess_signal(mean_rgb[:, i, 1], fps)
            rppg_rois.append(sig_roi)
        rppg_rois_arr = np.stack(rppg_rois, axis=1)
        rppg_avg = np.mean(rppg_rois_arr, axis=1)
        std_avg = np.std(rppg_avg)
        if std_avg > 0:
            rppg_avg = (rppg_avg - np.mean(rppg_avg)) / std_avg
        return rppg_avg, rppg_rois_arr
    signal_raw = mean_rgb[:, roi_idx, 1]
    return preprocess_signal(signal_raw, fps)

fps = metadata.get('video', {}).get('fps', 30.0)
rppg_green, green_rois = green_channel(mean_rgb, roi_idx=0, fps=fps, return_rois=True)
print(f'Tín hiệu rPPG thu được từ Green channel: {len(rppg_green)} mẫu.')

In [ ]:
# Cell 7 - Baseline 2: CHROM (Chrominance-based method)
def chrom_rppg(mean_rgb, fps, return_rois=False):
    """CHROM algorithm (de Haan & Jeanne, 2013) sử dụng overlap-add."""
    n_frames, n_rois, _ = mean_rgb.shape
    L = int(round(1.6 * fps))
    if L < 3:
        L = 3

    rppg_rois = []

    for roi_idx in range(n_rois):
        s_global = np.zeros(n_frames, dtype=np.float32)
        w_global = np.zeros(n_frames, dtype=np.float32)

        for i in range(n_frames - L + 1):
            window = mean_rgb[i : i + L, roi_idx, :]

            # Chuẩn hóa tạm thời
            mean_c = np.mean(window, axis=0)
            mean_c = np.where(mean_c == 0, 1.0, mean_c)
            c_norm = window / mean_c
            xs = 3 * c_norm[:, 0] - 2 * c_norm[:, 1]
            ys = 1.5 * c_norm[:, 0] + c_norm[:, 1] - 1.5 * c_norm[:, 2]
            std_xs = np.std(xs)
            std_ys = np.std(ys)

            if std_xs == 0 or std_ys == 0:
                s_win = np.zeros(L, dtype=np.float32)
            else:
                alpha = std_xs / std_ys
                s_win = xs - alpha * ys
            win = np.hanning(L)
            s_global[i : i + L] += s_win * win
            w_global[i : i + L] += win

        w_global = np.where(w_global == 0, 1.0, w_global)
        s_roi = s_global / w_global
        s_roi_filtered = preprocess_signal(s_roi, fps, detrend=True, normalize=True, bandpass=(0.7, 3.5))
        rppg_rois.append(s_roi_filtered)

    rppg_rois_arr = np.stack(rppg_rois, axis=1)
    rppg_avg = np.mean(rppg_rois_arr, axis=1)
    std_avg = np.std(rppg_avg)
    if std_avg > 0:
        rppg_avg = (rppg_avg - np.mean(rppg_avg)) / std_avg
    if return_rois:
        normalized_rois = np.zeros_like(rppg_rois_arr)
        for i in range(n_rois):
            std_i = np.std(rppg_rois_arr[:, i])
            if std_i > 0:
                normalized_rois[:, i] = (rppg_rois_arr[:, i] - np.mean(rppg_rois_arr[:, i])) / std_i
            else:
                normalized_rois[:, i] = rppg_rois_arr[:, i] - np.mean(rppg_rois_arr[:, i])
        return rppg_avg, normalized_rois
    return rppg_avg

rppg_chrom, chrom_rois = chrom_rppg(mean_rgb, fps, return_rois=True)
print(f'Tín hiệu rPPG thu được từ CHROM: {len(rppg_chrom)} mẫu.')

In [ ]:
# Cell 8 - Baseline 3: POS (Plane-Orthogonal-to-Skin method)
def pos_rppg(mean_rgb, fps, return_rois=False):
    """POS algorithm (Wang et al., 2017) sử dụng overlap-add."""
    n_frames, n_rois, _ = mean_rgb.shape
    L = int(round(1.6 * fps))
    if L < 3:
        L = 3
    rppg_rois = []
    P = np.array([[0, 1, -1], [-2, 1, 1]], dtype=np.float32)

    for roi_idx in range(n_rois):
        s_global = np.zeros(n_frames, dtype=np.float32)
        w_global = np.zeros(n_frames, dtype=np.float32)

        for i in range(n_frames - L + 1):
            window = mean_rgb[i : i + L, roi_idx, :]
            # Chuẩn hóa tạm thời
            mean_c = np.mean(window, axis=0)
            mean_c = np.where(mean_c == 0, 1.0, mean_c)
            c_norm = window / mean_c
            H_raw = P @ c_norm.T
            std_0 = np.std(H_raw[0])
            std_1 = np.std(H_raw[1])
            if std_1 == 0:
                s_win = np.zeros(L, dtype=np.float32)
            else:
                alpha = std_0 / std_1
                s_win = H_raw[0] + alpha * H_raw[1]
            win = np.hanning(L)
            s_global[i : i + L] += s_win * win
            w_global[i : i + L] += win

        w_global = np.where(w_global == 0, 1.0, w_global)
        s_roi = s_global / w_global
        s_roi_filtered = preprocess_signal(s_roi, fps, detrend=True, normalize=True, bandpass=(0.7, 3.5))
        rppg_rois.append(s_roi_filtered)

    rppg_rois_arr = np.stack(rppg_rois, axis=1)
    rppg_avg = np.mean(rppg_rois_arr, axis=1)
    std_avg = np.std(rppg_avg)
    if std_avg > 0:
        rppg_avg = (rppg_avg - np.mean(rppg_avg)) / std_avg
    if return_rois:
        normalized_rois = np.zeros_like(rppg_rois_arr)
        for i in range(n_rois):
            std_i = np.std(rppg_rois_arr[:, i])
            if std_i > 0:
                normalized_rois[:, i] = (rppg_rois_arr[:, i] - np.mean(rppg_rois_arr[:, i])) / std_i
            else:
                normalized_rois[:, i] = rppg_rois_arr[:, i] - np.mean(rppg_rois_arr[:, i])
        return rppg_avg, normalized_rois
    return rppg_avg

rppg_pos, pos_rois = pos_rppg(mean_rgb, fps, return_rois=True)
print(f'Tín hiệu rPPG thu được từ POS: {len(rppg_pos)} mẫu.')

In [ ]:
# Cell 9 - Ước lượng nhịp tim (Sliding Window FFT)
def estimate_hr(signal_in, fps, window_sec=30, step_sec=1, hr_range=(42, 210)):
    """Ước lượng nhịp tim (BPM) qua FFT cửa sổ trượt."""
    N = len(signal_in)
    W_size = int(round(window_sec * fps))
    S_step = int(round(step_sec * fps))
    if W_size > N:
        W_size = N
        S_step = N
    hr_trace = []
    timestamps_hr = []
    f_min, f_max = hr_range[0] / 60.0, hr_range[1] / 60.0
    start = 0
    while start + W_size <= N:
        end = start + W_size
        t_center = (start + end - 1) / (2.0 * fps)
        s_win = signal_in[start:end] - np.mean(signal_in[start:end])
        
        win = windows.hann(W_size)
        s_win = s_win * win
        
        fft_vals = np.fft.rfft(s_win)
        freqs = np.fft.rfftfreq(W_size, 1.0 / fps)
        power_spec = np.abs(fft_vals) ** 2
        
        valid_idx = (freqs >= f_min) & (freqs <= f_max)
        if np.sum(valid_idx) > 0:
            peak_idx = np.argmax(power_spec[valid_idx])
            peak_freq = freqs[valid_idx][peak_idx]
            hr = peak_freq * 60.0
        else:
            hr = np.nan
            
        hr_trace.append(hr)
        timestamps_hr.append(t_center)
        start += S_step
        if S_step <= 0:
            break
            
    return np.array(hr_trace, dtype=np.float32), np.array(timestamps_hr, dtype=np.float32)

print('Đang tính toán nhịp tim cho cả 3 baseline...')
hr_green, hr_ts = estimate_hr(rppg_green, fps, window_sec=WINDOW_SEC, step_sec=STEP_SEC)
hr_chrom, _ = estimate_hr(rppg_chrom, fps, window_sec=WINDOW_SEC, step_sec=STEP_SEC)
hr_pos, _ = estimate_hr(rppg_pos, fps, window_sec=WINDOW_SEC, step_sec=STEP_SEC)
print(f'Đã ước lượng nhịp tim. Số lượng cửa sổ ước lượng: {len(hr_green)}')

In [ ]:
# Cell 10 - Ground Truth Loader
def _estimate_running_hr_from_ppg(ppg, fps):
    """Ước lượng nhịp tim cục bộ từ tín hiệu raw PPG."""
    win_size = int(round(8.0 * fps))
    half_win = win_size // 2
    n = len(ppg)
    hr = np.zeros(n, dtype=np.float32)
    for i in range(n):
        start = max(0, i - half_win)
        end = min(n, i + half_win)
        w = ppg[start:end] - np.mean(ppg[start:end])
        if len(w) < 3:
            hr[i] = np.nan
            continue
        w = w * windows.hann(len(w))
        fft_vals = np.fft.rfft(w)
        freqs = np.fft.rfftfreq(len(w), 1.0 / fps)
        power = np.abs(fft_vals) ** 2
        valid_idx = (freqs >= 0.7) & (freqs <= 3.5)
        if np.sum(valid_idx) > 0:
            peak_idx = np.argmax(power[valid_idx])
            hr[i] = freqs[valid_idx][peak_idx] * 60.0
        else:
            hr[i] = np.nan
    return hr

def load_ground_truth(gt_path, fps_gt=None):
    """Nạp Ground Truth tự động nhận diện .txt hoặc .xmp"""
    gt_path = Path(gt_path)
    
    if gt_path.suffix.lower() == '.xmp':
        try:
            import xml.etree.ElementTree as ET
            tree = ET.parse(gt_path)
            root = tree.getroot()
            pulse_data, hr_data = [], []
            
            def find_elements(element):
                tag = element.tag.lower()
                if 'hr' in tag or 'heartrate' in tag or 'bpm' in tag:
                    text = element.text
                    if text:
                        nums = re.findall(r'[-+]?\d*\.\d+|\d+', text)
                        hr_data.extend([float(x) for x in nums])
                elif 'pulse' in tag or 'signal' in tag or 'wave' in tag or 'value' in tag or 'data' in tag:
                    text = element.text
                    if text:
                        nums = re.findall(r'[-+]?\d*\.\d+|\d+', text)
                        pulse_data.extend([float(x) for x in nums])
                for child in element:
                    find_elements(child)
            find_elements(root)
            
            if len(hr_data) > 0:
                gt_hr_arr = np.array(hr_data, dtype=np.float32)
                fps = fps_gt if fps_gt is not None else 1.0
                return gt_hr_arr, np.arange(len(gt_hr_arr)) / fps
            elif len(pulse_data) > 0:
                pulse_signal = np.array(pulse_data, dtype=np.float32)
                fps = fps_gt if fps_gt is not None else 30.0
                gt_hr_arr = _estimate_running_hr_from_ppg(pulse_signal, fps)
                return gt_hr_arr, np.arange(len(pulse_signal)) / fps
        except Exception:
            pass
            
    try:
        try:
            data = np.loadtxt(gt_path, delimiter=',')
        except Exception:
            data = np.loadtxt(gt_path)

        if data.ndim == 1:
            val = data
            fps = fps_gt if fps_gt is not None else 30.0
            ts = np.arange(len(val)) / fps
        else:
            if data.shape[1] == 2:
                mean_0 = np.nanmean(data[:, 0])
                mean_1 = np.nanmean(data[:, 1])
                if mean_0 > 40.0:
                    val = data[:, 0]
                    ts = data[:, 1]
                elif mean_1 > 40.0:
                    val = data[:, 1]
                    ts = data[:, 0]
                else:
                    val = data[:, 0]
                    ts = data[:, 1]
            elif data.shape[1] >= 3:
                val = data[:, 1]
                ts = data[:, 2]
            else:
                val = data[:, 0]
                ts = np.arange(len(val)) / (fps_gt if fps_gt is not None else 30.0)

        if np.nanmean(val) > 40.0:
            return val, ts
        else:
            fps = fps_gt if fps_gt is not None else 30.0
            gt_hr_arr = _estimate_running_hr_from_ppg(val, fps)
            return gt_hr_arr, ts
    except Exception as e:
        raise ValueError(f'Không thể đọc file ground truth: {e}')

# Nạp nhãn hoặc sinh nhãn giả demo
if GROUND_TRUTH_PATH.exists():
    gt_hr, gt_ts = load_ground_truth(GROUND_TRUTH_PATH, fps_gt=fps)
    print(f'Đã nạp Ground Truth thành công! Số mẫu: {len(gt_hr)}')
else:
    print(f'Cảnh báo: Không tìm thấy Ground Truth tại {GROUND_TRUTH_PATH}.')
    print('Đang sinh nhãn nhịp tim giả lập để hoàn thành luồng chạy...')
    gt_ts = hr_ts.copy()
    gt_hr = 70.0 + 3.0 * np.sin(gt_ts / 25.0) + np.random.normal(0, 1.0, size=len(gt_ts))

In [ ]:
# Cell 11 - Hàm tính toán Metrics đánh giá
def calculate_snr(signal, fps, gt_hr, band_width=0.1):
    sig_detrend = sig.detrend(signal - np.mean(signal))
    N = len(sig_detrend)
    if N < 3:
        return np.nan
    fft_vals = np.fft.rfft(sig_detrend)
    freqs = np.fft.rfftfreq(N, 1.0 / fps)
    power_spec = np.abs(fft_vals) ** 2
    f_gt = gt_hr / 60.0
    f_min, f_max = 0.7, 3.5
    fundamental_mask = (freqs >= (f_gt - band_width)) & (freqs <= (f_gt + band_width))
    harmonic_mask = (freqs >= (2 * f_gt - 2 * band_width)) & (freqs <= (2 * f_gt + 2 * band_width))
    signal_mask = fundamental_mask | harmonic_mask
    total_band_mask = (freqs >= f_min) & (freqs <= f_max)
    p_signal = np.sum(power_spec[signal_mask & total_band_mask])
    p_noise = np.sum(power_spec[(~signal_mask) & total_band_mask])
    if p_noise <= 0 or p_signal <= 0:
        return np.nan
    return 10 * np.log10(p_signal / p_noise)

def evaluate(pred_hr, pred_ts, gt_hr, gt_ts, rppg_signal=None, fps=30.0):
    """Đánh giá chất lượng dự đoán (MAE, RMSE, Pearson r, bias, SNR)"""
    if len(pred_hr) == 0 or len(gt_hr) == 0:
        return {'mae': np.nan, 'rmse': np.nan, 'pearson_r': np.nan, 'mean_error': np.nan, 'snr': np.nan}
    try:
        f_gt = interp1d(gt_ts, gt_hr, kind='linear', bounds_error=False, fill_value='extrapolate')
        gt_aligned = f_gt(pred_ts)
        mask = (~np.isnan(pred_hr)) & (~np.isnan(gt_aligned))
        if np.sum(mask) < 2:
            return {'mae': np.nan, 'rmse': np.nan, 'pearson_r': np.nan, 'mean_error': np.nan, 'snr': np.nan}
        p_c = pred_hr[mask]
        g_c = gt_aligned[mask]
        
        mae = np.mean(np.abs(p_c - g_c))
        rmse = np.sqrt(np.mean((p_c - g_c)**2))
        try:
            r, _ = pearsonr(p_c, g_c)
        except Exception:
            r = np.nan
        bias = np.mean(p_c - g_c)
        
        if rppg_signal is not None and len(g_c) > 0:
            snr = calculate_snr(rppg_signal, fps, np.nanmean(g_c))
        else:
            snr = np.nan
            
        return {'mae': float(mae), 'rmse': float(rmse), 'pearson_r': float(r), 'mean_error': float(bias), 'snr': float(snr)}
    except Exception as e:
        print(f'Lỗi: {e}')
        return {'mae': np.nan, 'rmse': np.nan, 'pearson_r': np.nan, 'mean_error': np.nan, 'snr': np.nan}

metrics_green = evaluate(hr_green, hr_ts, gt_hr, gt_ts, rppg_green, fps)
metrics_chrom = evaluate(hr_chrom, hr_ts, gt_hr, gt_ts, rppg_chrom, fps)
metrics_pos = evaluate(hr_pos, hr_ts, gt_hr, gt_ts, rppg_pos, fps)

import pandas as pd
metrics_df = pd.DataFrame({
    'Green Channel': metrics_green,
    'CHROM': metrics_chrom,
    'POS': metrics_pos
}).T
print('Bảng đánh giá kết quả các Baseline Methods:')
print(metrics_df.round(3))

In [ ]:
# Cell 12 - Trực quan hóa kết quả (Visualization)
fig, axes = plt.subplots(5, 1, figsize=(12, 20))

# 1. RGB
axes[0].plot(timestamps, mean_rgb[:, 0, 0], 'r', label='R (Trán)')
axes[0].plot(timestamps, mean_rgb[:, 0, 1], 'g', label='G (Trán)')
axes[0].plot(timestamps, mean_rgb[:, 0, 2], 'b', label='B (Trán)')
axes[0].set_title('1. Chuỗi thời gian màu sắc RGB trung bình vùng trán (Forehead ROI)')
axes[0].set_ylabel('Giá trị pixel')
axes[0].legend()
axes[0].grid(True)

# 2. rPPG signals
t_start, t_end = min(timestamps), min(timestamps) + 15.0
mask_plot = (timestamps >= t_start) & (timestamps <= t_end)
axes[1].plot(timestamps[mask_plot], rppg_green[mask_plot], label='Green Channel')
axes[1].plot(timestamps[mask_plot], rppg_chrom[mask_plot], label='CHROM')
axes[1].plot(timestamps[mask_plot], rppg_pos[mask_plot], label='POS')
axes[1].set_title('2. Đoạn tín hiệu rPPG chuẩn hóa (15 giây đầu)')
axes[1].set_ylabel('Biên độ chuẩn hóa')
axes[1].legend()
axes[1].grid(True)

# 3. Phổ tần số (Frequency Spectrum)
for label, sig_data in [('Green Channel', rppg_green), ('CHROM', rppg_chrom), ('POS', rppg_pos)]:
    sig_detrend = sig.detrend(sig_data - np.mean(sig_data))
    fft_v = np.fft.rfft(sig_detrend)
    freqs_v = np.fft.rfftfreq(len(sig_detrend), 1.0 / fps)
    mag_v = np.abs(fft_v)
    if np.max(mag_v) > 0:
        mag_v = mag_v / np.max(mag_v)
    axes[2].plot(freqs_v * 60.0, mag_v, label=label)

mean_gt = np.nanmean(gt_hr)
axes[2].axvline(mean_gt, color='k', linestyle='--', label=f'GT trung bình ({mean_gt:.1f} BPM)', linewidth=2)
axes[2].set_xlim(40, 220)
axes[2].set_title('3. Phổ tần số chuẩn hóa của tín hiệu rPPG (FFT Magnitude)')
axes[2].set_xlabel('Tần số (BPM)')
axes[2].set_ylabel('Biên độ phổ chuẩn hóa')
axes[2].legend()
axes[2].grid(True)

# 4. HR Comparison
axes[3].plot(gt_ts, gt_hr, 'k--', label='Ground Truth', linewidth=2)
axes[3].plot(hr_ts, hr_green, label='Green Channel', alpha=0.7)
axes[3].plot(hr_ts, hr_chrom, label='CHROM', alpha=0.7)
axes[3].plot(hr_ts, hr_pos, label='POS', alpha=0.7)
axes[3].set_title('4. So sánh nhịp tim ước lượng và nhãn gốc')
axes[3].set_xlabel('Thời gian (giây)')
axes[3].set_ylabel('Nhịp tim (BPM)')
axes[3].legend()
axes[3].grid(True)

# 5. Bar chart of metrics
metrics_names = ['mae', 'rmse', 'snr']
x = np.arange(len(metrics_names))
width = 0.25
axes[4].bar(x - width, [metrics_green[m] for m in metrics_names], width, label='Green')
axes[4].bar(x, [metrics_chrom[m] for m in metrics_names], width, label='CHROM')
axes[4].bar(x + width, [metrics_pos[m] for m in metrics_names], width, label='POS')
axes[4].set_title('5. So sánh chỉ số sai số (MAE, RMSE) và SNR')
axes[4].set_xticks(x)
axes[4].set_xticklabels(['MAE (BPM)', 'RMSE (BPM)', 'SNR (dB)'])
axes[4].legend()
axes[4].grid(True)

plt.tight_layout()
video_name = VIDEO_PATH.stem if VIDEO_PATH else 'sample'
plot_save = OUTPUT_DIR / f'{video_name}_visualization.png'
plt.savefig(plot_save, dpi=150)
print(f'Đã lưu hình trực quan tại: {plot_save}')
plt.show()

In [ ]:
# Cell 13 - Lưu tệp kết quả (Save outputs)
video_name = VIDEO_PATH.stem if VIDEO_PATH else 'sample'

# Lưu mảng RGB
np.savez_compressed(
    OUTPUT_DIR / f'{video_name}_rgb_signals.npz',
    mean_rgb=mean_rgb,
    timestamps=timestamps,
    valid_frame_indices=npz_data['frame_indices'][npz_data['valid']] if 'frame_indices' in npz_data else np.arange(len(timestamps))
)

# Lưu tín hiệu rPPG (bao gồm cả tín hiệu trung bình và tín hiệu per-ROI)
np.savez_compressed(
    OUTPUT_DIR / f'{video_name}_rppg_signals.npz',
    green=rppg_green,
    chrom=rppg_chrom,
    pos=rppg_pos,
    green_roi=green_rois,
    chrom_roi=chrom_rois,
    pos_roi=pos_rois,
    timestamps=timestamps
)

# Lưu nhịp tim ước lượng
np.savez_compressed(
    OUTPUT_DIR / f'{video_name}_hr_traces.npz',
    green_hr=hr_green,
    chrom_hr=hr_chrom,
    pos_hr=hr_pos,
    gt_hr=gt_hr,
    hr_timestamps=hr_ts
)

# Lưu file JSON metrics
metrics_output = {
    'green': metrics_green,
    'chrom': metrics_chrom,
    'pos': metrics_pos
}
with open(OUTPUT_DIR / f'{video_name}_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_output, f, indent=2)
    
print(f'Đã lưu toàn bộ file kết quả vào: {OUTPUT_DIR}')

In [ ]:
# Cell 14 - Xử lý hàng loạt trên tập dữ liệu (Batch mode)
PROCESS_ALL_VIDEOS = os.environ.get('PROCESS_ALL_VIDEOS', '0') == '1'

if PROCESS_ALL_VIDEOS:
    print('Chế độ Batch Mode đã bật. Đang quét thư mục outputs/task1_preprocessing/...')
    manifest_path = Path('outputs/task1_preprocessing/task1_dataset_manifest.json')
    if not manifest_path.exists():
        print('Lỗi: Không tìm thấy file manifest của Task 1.')
    else:
        with open(manifest_path, 'r', encoding='utf-8') as f:
            manifest = json.load(f)
        results = manifest.get('results', [])
        processed_vids = [r for r in results if r.get('status') == 'processed']
        print(f'Tìm thấy {len(processed_vids)} video cần xử lý.')
        
        batch_metrics = {}
        for entry in tqdm(processed_vids, desc='Batch Processing Task 2'):
            v_path = Path(entry['video'])
            v_npz = Path(entry['arrays'])
            v_meta = Path(entry['metadata'])
            
            v_path = (PROJECT_ROOT / v_path).resolve() if not v_path.is_absolute() else v_path.resolve()
            v_npz = (PROJECT_ROOT / v_npz).resolve() if not v_npz.is_absolute() else v_npz.resolve()
            v_meta = (PROJECT_ROOT / v_meta).resolve() if not v_meta.is_absolute() else v_meta.resolve()
            
            v_split = entry.get('split', 'UNSORTED')
            v_subj = entry.get('subject', 'unknown')
            v_out_dir = OUTPUT_DIR / v_split / v_subj
            v_out_dir.mkdir(parents=True, exist_ok=True)
            
            v_gt = v_path.parent / 'ground_truth.txt'
            if not v_gt.exists():
                v_gt = v_path.parent / 'gtdump.xmp'
                
            try:
                data_npz = dict(np.load(v_npz, allow_pickle=True))
                with open(v_meta, 'r', encoding='utf-8') as fm:
                    meta_data = json.load(fm)
                v_fps = meta_data['video']['fps']
                
                if v_path.exists():
                    m_rgb, ts = extract_roi_rgb(v_path, data_npz, meta_data, progress_bar=False)
                else:
                    continue
                    
                r_green, r_green_rois = green_channel(m_rgb, roi_idx=0, fps=v_fps, return_rois=True)
                r_chrom, r_chrom_rois = chrom_rppg(m_rgb, fps=v_fps, return_rois=True)
                r_pos, r_pos_rois = pos_rppg(m_rgb, fps=v_fps, return_rois=True)
                
                h_green, h_ts = estimate_hr(r_green, v_fps, window_sec=WINDOW_SEC, step_sec=STEP_SEC)
                h_chrom, _ = estimate_hr(r_chrom, v_fps, window_sec=WINDOW_SEC, step_sec=STEP_SEC)
                h_pos, _ = estimate_hr(r_pos, v_fps, window_sec=WINDOW_SEC, step_sec=STEP_SEC)
                
                if v_gt.exists():
                    g_hr, g_ts = load_ground_truth(v_gt, fps_gt=v_fps)
                    m_green = evaluate(h_green, h_ts, g_hr, g_ts, r_green, v_fps)
                    m_chrom = evaluate(h_chrom, h_ts, g_hr, g_ts, r_chrom, v_fps)
                    m_pos = evaluate(h_pos, h_ts, g_hr, g_ts, r_pos, v_fps)
                    
                    metrics_res = {'green': m_green, 'chrom': m_chrom, 'pos': m_pos}
                    batch_metrics[str(v_path)] = metrics_res
                    
                    with open(v_out_dir / f'{v_path.stem}_metrics.json', 'w', encoding='utf-8') as f_met:
                        json.dump(metrics_res, f_met, indent=2)
                        
                np.savez_compressed(v_out_dir / f'{v_path.stem}_rgb_signals.npz', mean_rgb=m_rgb, timestamps=ts)
                np.savez_compressed(
                    v_out_dir / f'{v_path.stem}_rppg_signals.npz',
                    green=r_green,
                    chrom=r_chrom,
                    pos=r_pos,
                    green_roi=r_green_rois,
                    chrom_roi=r_chrom_rois,
                    pos_roi=r_pos_rois,
                    timestamps=ts
                )
                
            except Exception as e:
                print(f'Lỗi xử lý video {v_path.name}: {e}')
                
        if batch_metrics:
            print('\n=== KẾT QUẢ ĐÁNH GIÁ TRUNG BÌNH TOÀN DATASET ===')
            for method in ['green', 'chrom', 'pos']:
                maes = [m[method]['mae'] for m in batch_metrics.values() if not np.isnan(m[method]['mae'])]
                rmses = [m[method]['rmse'] for m in batch_metrics.values() if not np.isnan(m[method]['rmse'])]
                snrs = [m[method]['snr'] for m in batch_metrics.values() if not np.isnan(m[method]['snr'])]
                print(f'  * {method.upper()}:')
                print(f'    - MAE: {np.mean(maes):.3f} ± {np.std(maes):.3f} BPM')
                print(f'    - RMSE: {np.mean(rmses):.3f} ± {np.std(rmses):.3f} BPM')
                if snrs:
                    print(f'    - SNR: {np.mean(snrs):.3f} ± {np.std(snrs):.3f} dB')
else:
    print('Batch mode đang tắt. Thiết lập PROCESS_ALL_VIDEOS=1 để chạy hàng loạt.')